In [1]:
!pip install gseapy mygene --quiet


In [2]:
import warnings
warnings.filterwarnings('ignore')

import gseapy as gp
import mygene
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Enrichment Analysis — ORA (Overrepresentation Analysis)

GSEA prerank wymaga rankingu **wszystkich** ~20 000 genow w genomie.
Nasz stability ranking zawiera tylko ~700 genow po feature selection — za malo na GSEA.

Zamiast tego uzywamy **ORA (Enrichr)** — testuje, czy wsrod naszych top genow
nadreprezentowane sa konkretne sciezki biologiczne w porownaniu do tla.


In [3]:
# ── wczytaj stability ranking ────────────────────────────────────
df = pd.read_csv('gene_stability_kfold.csv', index_col='rank')

# zostaw tylko geny z fold_count > 0
df = df[df['fold_count'] > 0].copy()
df['ranking_score'] = df['fold_count'] * df['mean_score']
df = df.sort_values('ranking_score', ascending=False)

ensembl_ids = df['gene'].tolist()
print(f'Geny z fold_count > 0: {len(df)}')
print(f'Przyklad: {ensembl_ids[:5]}')


FileNotFoundError: [Errno 2] No such file or directory: 'gene_stability_kfold.csv'

In [ ]:
# ── konwersja Ensembl ID -> symbol genu ─────────────────────────
mg = mygene.MyGeneInfo()
result = mg.querymany(
    ensembl_ids,
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True,
    verbose=False,
)

result = result.dropna(subset=['symbol'])
result = result[~result.index.duplicated(keep='first')]

df = df.set_index('gene')
df['symbol'] = result['symbol']
df = df.dropna(subset=['symbol'])
df = df[~df['symbol'].duplicated(keep='first')]

print(f'Geny po konwersji: {len(df)}')
print(df[['symbol', 'fold_count', 'mean_score', 'ranking_score']].head(10))


## ORA — Overrepresentation Analysis

Bierzemy top geny (np. top-50, top-100) i testujemy, ktore sciezki biologiczne
sa w nich nadreprezentowane. Enrichr korzysta z testu Fishera.


In [ ]:
# ── ORA (Enrichr) dla roznych rozmiarow top-N ───────────────────
GENE_SETS = [
    'MSigDB_Hallmark_2020',
    'KEGG_2021_Human',
    'Reactome_2022',
]

TOP_N_LIST = [30, 50, 100, 200]
all_results = []

for top_n in TOP_N_LIST:
    gene_list = df.head(top_n)['symbol'].tolist()
    print(f'\n--- ORA top-{top_n} ({len(gene_list)} genow) ---')

    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=GENE_SETS,
        organism='human',
        outdir=f'./enrichr_top{top_n}/',
        no_plot=True,
    )

    res = enr.results.copy()
    res['top_n'] = top_n
    sig = res[res['Adjusted P-value'] < 0.05]
    print(f'  Istotne sciezki (adj p < 0.05): {len(sig)}')
    if len(sig) > 0:
        print(sig[['Term', 'Adjusted P-value', 'Overlap', 'Combined Score']].head(10).to_string())
    all_results.append(res)

ora_df = pd.concat(all_results, ignore_index=True)
print(f'\nLacznie wynikow: {len(ora_df)}')


In [ ]:
# ── najlepsze wyniki ze wszystkich progów ────────────────────────
significant = ora_df[ora_df['Adjusted P-value'] < 0.05].copy()
significant = significant.sort_values('Adjusted P-value')

# usun duplikaty sciezek (zachowaj najlepsza wersje)
best = significant.drop_duplicates(subset='Term', keep='first')

print(f'Istotne unikalne sciezki: {len(best)}')
if len(best) > 0:
    print(best[['Term', 'Adjusted P-value', 'Overlap', 'Combined Score', 'top_n', 'Genes']].head(20).to_string())
else:
    # pokaz najblizsze istotnosci
    print('Brak sciezek z adj p < 0.05. Najblizsze:')
    near = ora_df.sort_values('Adjusted P-value').drop_duplicates(subset='Term', keep='first')
    print(near[['Term', 'Adjusted P-value', 'Overlap', 'Combined Score', 'top_n']].head(20).to_string())


In [ ]:
# ── wykres top sciezek ──────────────────────────────────────────
show = best.head(20) if len(best) > 0 else ora_df.sort_values('Adjusted P-value').drop_duplicates('Term').head(20)
show = show.copy()
show['neg_log_p'] = -np.log10(show['Adjusted P-value'] + 1e-300)
show = show.sort_values('neg_log_p')

fig, ax = plt.subplots(figsize=(10, max(5, len(show) * 0.4)))
bars = ax.barh(show['Term'], show['neg_log_p'], color='#6366f1', edgecolor='white')
ax.axvline(-np.log10(0.05), color='red', ls='--', alpha=0.7, label='adj p = 0.05')
ax.set_xlabel('-log10(Adjusted P-value)')
ax.set_title('Top enriched pathways (ORA / Enrichr)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── zapis ───────────────────────────────────────────────────────
ora_df.to_csv('enrichr_all_results.csv', index=False)
if len(best) > 0:
    best.to_csv('enrichr_significant.csv', index=False)
    print('Zapisano: enrichr_all_results.csv, enrichr_significant.csv')
else:
    print('Zapisano: enrichr_all_results.csv')
